In [48]:
import os
import glob
from pathlib import Path
import numpy as np
import librosa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
def extract_features(file_path, sr=16000, n_mfcc=20):
    y, sr = librosa.load(file_path, sr=sr)
    y = librosa.util.normalize(y)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)

    features = np.concatenate([mfcc_mean, mfcc_std])

    return features

In [9]:
DATA_DIR = 'datasets/b_vs_p/'

X = []
y = []

for label_name, label in [("barbie", 0), ('puppy', 1)]:
    files = glob.glob(os.path.join(DATA_DIR, label_name, '*wav'))
    for f in files:
        feat = extract_features(f)
        X.append(feat)
        y.append(label)

X = np.array(X)
y = np.array(y)

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scales = scaler.transform(X_test)

In [ ]:
log_reg = LogisticRegression(
    C=1.0, penalty="l2", solver='liblinear', class_weight='balanced', random_state=42
)
log_reg.fit(X_train_scaled, y_train)

y_test_pred = log_reg.predict(X_test_scales)
print("LogReg validation:")
print(classification_report(y_test, y_test_pred))

LogReg validation:
              precision    recall  f1-score   support

           0       1.00      0.70      0.82        10
           1       0.77      1.00      0.87        10

    accuracy                           0.85        20
   macro avg       0.88      0.85      0.85        20
weighted avg       0.88      0.85      0.85        20



____
____

In [91]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset, TensorDataset
from audiomentations import Compose, AddGaussianNoise, PitchShift, Shift

from tqdm import tqdm

In [46]:
augment_transform = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.01, p=0.5),
    PitchShift(min_semitones=-1, max_semitones=1, p=0.5),
    Shift(p=0.5),
])

In [ ]:
class BarbiePuppyDataset(Dataset):
    def __init__(self, files, labels, sr=16000, n_mfcc=20, augment=True):
        self.files = files
        self.labels = labels
        self.sr = sr
        self.n_mfcc=n_mfcc
        self.augment = augment

    def __len__(self):
        return len(self.files)
    
    def _extract_features(self, y):
        y = librosa.util.normalize(y)
        mfcc = librosa.feature.mfcc(y=y, sr=self.sr, n_mfcc=self.n_mfcc)
        mfcc_mean = mfcc.mean(axis=1)
        mfcc_std = mfcc.std(axis=1)
        feats = np.concatenate([mfcc_mean, mfcc_std], axis=0)
        return feats

    def __getitem__(self, idx):
        path = self.files[idx]
        label = self.labels[idx]

        y, sr = librosa.load(path)

        if self.augment is True:
            y = augment_transform(samples=y, sample_rate=self.sr)

        feats = self._extract_features(y)

        return feats, np.float32(label)

In [165]:
# функуия для разбиения данных
def split_dataset(root_dir, test_size=0.2, val_size=0.2):
    root_path = Path(root_dir)
    files = list(root_path.rglob("*.wav"))
    files = [str(p) for p in files]
    labels = [1 if "barbie" in f else 0 for f in files]

    X_train, X_temp, y_train, y_temp = train_test_split(
        files, labels, test_size=test_size + val_size, stratify=labels, random_state=42
    )

    rel_val = val_size / (test_size + val_size)

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=rel_val, stratify=y_temp, random_state=42
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

In [90]:
def collect_numpy(ds, batch_size=32):
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    X_list, y_list = [], []
    for feats, labels in dl:
        X_list.append(feats.numpy())
        y_list.append(labels.numpy())
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X, y

In [166]:
train_files, train_labels, val_files, val_labels, test_files, test_lables = split_dataset(
    DATA_DIR
)

train_ds_orig = BarbiePuppyDataset(train_files, train_labels, augment=False)
train_ds_aug_1 = BarbiePuppyDataset(train_files, train_labels)
train_ds_aug_2 = BarbiePuppyDataset(train_files, train_labels)

train_ds_np = ConcatDataset([train_ds_orig, train_ds_aug_1, train_ds_aug_2])

val_ds_np = BarbiePuppyDataset(val_files, val_labels, augment=False)
test_ds_np = BarbiePuppyDataset(test_files, test_lables, augment=False)

X_train_raw, y_train = collect_numpy(train_ds_np)
X_val_raw, y_val = collect_numpy(val_ds_np)
X_test_raw,  y_test  = collect_numpy(test_ds_np)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t,   y_val_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)

train_dl = DataLoader(train_ds, batch_size=8, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False)
test_dl = DataLoader(test_ds, batch_size=8, shuffle=False)

In [ ]:
input_dim = X_train.shape[1]

class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

In [219]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = LogisticRegressionModel(input_dim).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [222]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for feats, labels in tqdm(train_dl, desc=f'Train: epoch {epoch + 1}'):
        feats = feats.to(device)
        labels = labels.to(device).unsqueeze(1)

        logists = model(feats)
        loss = criterion(logists, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * feats.size(0)

    avg_loss = total_loss / len(train_dl.dataset)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for feats, labels in tqdm(val_dl, desc=f'Val epoch {epoch + 1}'):
            feats = feats.to(device)
            labels = labels.to(device).unsqueeze(1)

            logits = model(feats)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.numel()

    val_acc = correct / total        
    print(f'    train_loss={avg_loss:.4f}, val_acc={val_acc:.4f}')

Val epoch 1: 100%|██████████| 3/3 [00:00<00:00, 979.60it/s]


    train_loss=0.5608, val_acc=0.7000


Val epoch 2: 100%|██████████| 3/3 [00:00<00:00, 636.18it/s]


    train_loss=0.5499, val_acc=0.7000


Val epoch 3: 100%|██████████| 3/3 [00:00<00:00, 1525.57it/s]


    train_loss=0.5398, val_acc=0.7000


Val epoch 4: 100%|██████████| 3/3 [00:00<00:00, 1094.93it/s]


    train_loss=0.5313, val_acc=0.7000


Val epoch 5: 100%|██████████| 3/3 [00:00<00:00, 897.50it/s]

    train_loss=0.5240, val_acc=0.7000


In [223]:
model.eval()
all_labels = []
all_preds = []

with torch.no_grad():
    for feats, labels in tqdm(test_dl):
        feats = feats.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(feats)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        all_labels.append(labels.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

y_true = np.concatenate(all_labels).ravel()
y_pred = np.concatenate(all_preds).ravel()

print(classification_report(y_true, y_pred))

100%|██████████| 3/3 [00:00<00:00, 428.00it/s]

              precision    recall  f1-score   support

         0.0       0.82      0.90      0.86        10
         1.0       0.89      0.80      0.84        10

    accuracy                           0.85        20
   macro avg       0.85      0.85      0.85        20
weighted avg       0.85      0.85      0.85        20

